# Importation de packages

In [ ]:
import os
import pandas as pd
from utils import charger_donnees_api

# Données

On définit le dossier de données qui n'est créé que s'il n'existe pas encore dans le repertoire de travail.

In [ ]:
DOSSIER_DONNEES = "data/raw"
os.makedirs(DOSSIER_DONNEES, exist_ok=True)

Nous importons ici les données sur les pollens via une l'api open-meteo (**air-quality** pour les données sur les pollens et **archives** pour les données météo); nous avons considéré les données sur un an du 1er janvier 2023 au 01 janvier 2024 pour Rennes.

Données Pollen

In [ ]:
url_pollen = "https://air-quality-api.open-meteo.com/v1/air-quality"

params_pollen = {
    "latitude": 48.11,
    "longitude": -1.67,
    "hourly": ["birch_pollen", "grass_pollen"],
    "start_date": "2023-01-01",
    "end_date": "2023-12-31"
}

fichier_pollen = os.path.join(DOSSIER_DONNEES, "pollen.csv")


df_pol = charger_donnees_api(
    url=url_pollen,
    params=params_pollen,
    fichier_cache=fichier_pollen,
    force_reload= True
)


Données météo

In [ ]:
url_meteo = "https://archive-api.open-meteo.com/v1/archive"

params_meteo = {
    "latitude": 48.11,
    "longitude": -1.67,
    "hourly": ["temperature_2m", "precipitation", "wind_speed_10m"],
    "start_date": "2023-01-01",
    "end_date": "2023-12-31",
    "timezone": "Europe/Paris"
}

fichier_meteo = os.path.join(DOSSIER_DONNEES, "meteo.csv")


df_met = charger_donnees_api(
    url=url_meteo,
    params=params_meteo,
    fichier_cache=fichier_meteo,
    force_reload= True
)

In [ ]:
df_meteo = df_met.copy()
df_pollen = df_pol.copy()

In [ ]:
pd.concat(
    [df_pollen.head(), df_meteo.head()],
    axis=1,
    keys=["Pollen", "Meteo"]
)

Pour faciliter la lisibilité et la manipulation des variables, il est nécessaire de les renommer.

In [ ]:
df_pollen = df_pollen.rename(columns={
    "birch_pollen": "pollen_bouleau",
    "grass_pollen": "pollen_graminees"
})

df_meteo = df_meteo.rename(columns={
    "temperature_2m": "temperature",
    "precipitation": "precipitations",
    "wind_speed_10m": "vitesse_vent"
})

Pour répondre à la thématique, nous devons fusionner les deux bases; Mais avant  toute fusion, il faut vérifier la structure des deux bases et surtout étudier la clé de fusion qui ici est la date.

Afin de garantir l'intégrité de nos deux bases suivant qui sont temporelles, nous vérifions la régularité du pas de temps via le calcul des différences successives (.diff()). Cette étape confirme la continuité des observations horaires et l'absence de dates manquantes.

In [ ]:
pd.concat(
    [df_pollen["date"].diff().value_counts().head(), df_meteo["date"].diff().value_counts().head()],
    axis=1,
    keys=["Pollen", "Meteo"]
)

Vérifions que les périodes se chevauchent

In [ ]:
pd.DataFrame({
    "Pollen": [df_pollen["date"].min(), df_pollen["date"].max()],
    "Meteo": [df_meteo["date"].min(), df_meteo["date"].max()]
}, index=["Min date", "Max date"])

On remarque que les données des deux bases sont sans fuseau attachés, ce qui facilite la fusion locale.

In [ ]:
print(df_meteo["date"].dt.tz)
print(df_pollen["date"].dt.tz)

Fusion des deux bases: avec toutes les vérifications qui viennent d'être faites sur la variable date qui est la clé primaire ici, on  peut fusionner simplement avec un inner_join

In [ ]:
df_pollen_meteo = pd.merge(df_pollen, df_meteo, on="date", how="inner")

df_pollen_meteo.to_csv("data/raw/df_pollen_meteo_merge.csv")

print(df_pollen_meteo.shape)
df_pollen_meteo.head()

In [ ]:
pd.concat(
    [df_pollen.isnull().sum(), df_meteo.isnull().sum()],
    axis=1,
    keys=["Pollen", "Meteo"]
)


In [ ]:
pd.concat(
    [df_pollen.describe(), df_meteo.describe()],
    axis=1,
    keys=["Pollen", "Meteo"]
)